# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muneeb-khokhar/flyrank-ml-track/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Per training-honest-models, my label shape is "yes/no with an observed label" (is_declining), so I start with Logistic Regression (readable, gives a coefficient story) then Random Forest (stronger, handles nonlinear signal combinations my baseline rule couldn't). Both are evaluated as ranking scores at Precision@50, since the actual decision this supports is "which pages first" — same framing as my w04 baseline.

In [2]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, pandas as pd
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "read_parquet(['hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet', 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'])"
REL_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
REL_QUERY   = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"

data = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {REL}),
    agg AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks      ELSE 0 END) AS clk_prev30,
               AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.gsc_avg_position > 0
                        THEN f.gsc_avg_position END)                                                     AS pos_prev30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY AND f.gsc_impressions > 0
                        THEN 1 ELSE 0 END)                                                               AS days_active_prev30
        FROM {REL} f, bounds b
        GROUP BY 1,2
        HAVING imp_prev30 >= 10
    ),
    dated AS (
        SELECT a.*, DATE_DIFF('day', c.content_updated_date, (SELECT end_d FROM bounds)) AS days_since_update
        FROM agg a JOIN {REL_CONTENT} c ON a.content_hash_id = c.content_hash_id
    ),
    qsig AS (
        SELECT content_hash_id,
               ANY_VALUE(rare_impressions_share)       AS rare_share,
               ANY_VALUE(anonymized_impressions_share) AS anon_share,
               SUM(impressions_90d)                    AS kept_impressions
        FROM {REL_QUERY} GROUP BY content_hash_id
    )
    SELECT d.*, q.rare_share, q.anon_share,
           (d.imp_last30 < 0.8 * d.imp_prev30) AS is_declining
    FROM dated d LEFT JOIN qsig q ON d.content_hash_id = q.content_hash_id
""").df()

print(f"{len(data):,} rows | decline rate: {data['is_declining'].mean():.3f}")
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,507 rows | decline rate: 0.281


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_prev30,pos_prev30,days_active_prev30,days_since_update,rare_share,anon_share,is_declining
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,315.0,0.0,13.343549,29.0,-57,0.140230,0.818391,False
1,client_e547b89c05043229,content_9abd8b303f805847,14525.0,744.0,6.0,6.495085,29.0,-83,0.093750,0.643072,False
2,client_e547b89c05043229,content_5f58c55cbfee172a,378.0,523.0,0.0,10.771976,29.0,-57,0.081583,0.703554,True
3,client_e547b89c05043229,content_6fe390ba3af1e456,4509.0,3119.0,3.0,38.846926,29.0,-56,0.027216,0.752529,False
4,client_e547b89c05043229,content_3ad5d2160242b9ca,979.0,995.0,2.0,9.556644,29.0,-56,0.087969,0.858344,False


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

GroupShuffleSplit on client_hash_id — same as w03/w04. Pages within one client correlate strongly (shared site health, shared algorithm impact), so a random row-level split would let the model see a client's "twin" pages in training and test, inflating the score. Grouping by client is the only honest split for this question.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'days_active_prev30',
                'days_since_update', 'rare_share', 'anon_share']
model_df = data.dropna(subset=feature_cols).reset_index(drop=True)

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))

X, y = model_df[feature_cols], model_df['is_declining']
X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

print(f"Train: {len(X_tr):,} rows | Test: {len(X_te):,} rows | Test base rate: {y_te.mean():.3f}")

# Baseline rule, recomputed on THIS exact test split (same rule as w04)
te_df = model_df.iloc[te_idx]
baseline_score = ((te_df['days_since_update'] >= 180).astype(int) *
                   (te_df['imp_prev30'] >= 500).astype(int) * te_df['imp_prev30'])

Train: 59,836 rows | Test: 21,610 rows | Test base rate: 0.161


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same test split, same Precision@50 metric, both models plus the recomputed baseline in one table.

In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

scaler = StandardScaler().fit(X_tr)
logreg = LogisticRegression(max_iter=1000, random_state=42).fit(scaler.transform(X_tr), y_tr)
logreg_scores = logreg.predict_proba(scaler.transform(X_te))[:, 1]

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
rf_scores = rf.predict_proba(X_te)[:, 1]

results = pd.DataFrame({
    'method': ['base_rate (random)', 'baseline_rule (w04)', 'logistic_regression', 'random_forest'],
    'precision_at_50': [
        y_te.mean(),
        precision_at_k(baseline_score, y_te, 50),
        precision_at_k(logreg_scores, y_te, 50),
        precision_at_k(rf_scores, y_te, 50),
    ]
})
print(results.to_string(index=False))

             method  precision_at_50
 base_rate (random)         0.161499
baseline_rule (w04)         0.180000
logistic_regression         0.260000
      random_forest         0.620000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Permutation importance shows what the model actually leans on — then I sanity-check whether the top feature makes sense or looks suspiciously perfect (a signal of leakage).

The model leans almost entirely on imp_prev30 — pages with more existing traffic show clearer signal in either direction, which makes sense (more data points = less noise). rare_share (how much of a page's traffic comes from long-tail queries) is a distant second — plausibly relevant since long-tail-heavy pages may be more volatile. days_active_prev30 contributed essentially nothing (-0.001), suggesting simple day-count consistency doesn't add real signal once impression volume is already known. The confident-but-wrong cases (below) show the model over-trusting high-volume pages that turned out stable — likely pages with naturally noisy week-to-week traffic that crossed the 20% threshold by chance rather than real decline.

At an 0.8 confidence threshold, the model had zero false positives in this test split — a clean result. Lowering to 0.5 surfaces 3 borderline wrong cases, and all three share something specific: a negative days_since_update, meaning these pages were actually updated after my feature window ended (their content_updated_date falls past the March cutoff). The model still flagged them as "declining" based on their high volume (1,267–3,267 prev-30 impressions), but they turned out stable — plausibly because a real content update genuinely fixed whatever was causing the dip, which my features can't see since days_since_update becomes meaningless (even misleading) for content edited outside the observed window. That's a concrete limitation of this feature, not just this model.

In [6]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=42, n_jobs=-1)
importances = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)
print("Permutation importance:")
print(importances)

# 3 concrete wrong cases: high-confidence predictions that were actually wrong
te_df = te_df.copy()
te_df['rf_score'] = rf_scores
te_df['actual'] = y_te.values
wrong_confident = te_df[(te_df['rf_score'] > 0.8) & (te_df['actual'] == False)] \
                    .sort_values('rf_score', ascending=False).head(3)
print("\n3 confident-but-wrong cases (model said 'declining', actually stable):")
wrong_confident[['client_hash_id', 'content_hash_id', 'rf_score', 'imp_prev30', 'days_since_update', 'pos_prev30']]

Permutation importance:
imp_prev30            0.046886
rare_share            0.013619
pos_prev30            0.004961
days_since_update     0.003332
clk_prev30            0.002971
anon_share            0.000620
days_active_prev30   -0.001092
dtype: float64

3 confident-but-wrong cases (model said 'declining', actually stable):


,client_hash_id,content_hash_id,rf_score,imp_prev30,days_since_update,pos_prev30


In [7]:
print(f"Number of confident-but-wrong cases (threshold 0.8): {len(wrong_confident)}")

# If empty, lower the threshold to find real borderline examples instead
if len(wrong_confident) == 0:
    wrong_confident_relaxed = te_df[(te_df['rf_score'] > 0.5) & (te_df['actual'] == False)] \
        .sort_values('rf_score', ascending=False).head(3)
    print(f"\nAt a lower 0.5 threshold, found {len(wrong_confident_relaxed)} cases instead:")
    print(wrong_confident_relaxed[['client_hash_id', 'content_hash_id', 'rf_score', 'imp_prev30', 'days_since_update', 'pos_prev30']])

Number of confident-but-wrong cases (threshold 0.8): 0

At a lower 0.5 threshold, found 3 cases instead:
                client_hash_id           content_hash_id  rf_score  \
70575  client_ff644d8251367cbb  content_e3eece1c9c54d5a7  0.746667   
30166  client_ff644d8251367cbb  content_c7ef90df57c40cf3  0.743333   
34068  client_23a62021009f63c4  content_7b312163c4471c35  0.736667   

       imp_prev30  days_since_update  pos_prev30  
70575      1267.0                -48    6.123814  
30166      3267.0                -48    3.754383  
34068      1468.0                -94   10.991758  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.